# 이마트 쇼핑몰 리뷰 요약 화면 만들기
### Kiwi 형태소 분석 + KNU 한국어 감성사전

**환경:** Google Colab / Jupyter Notebook (인터넷 연결 필요)
**준비물:** `TS_2-4.남성화장품.xlsx` (상품평 3,001건)
**소요 시간:** 약 40분

## 오늘 만들 결과
쇼핑몰에서 자주 보는 **리뷰 요약 화면**을 직접 만듭니다.

| 화면 요소 | 무엇을 보여 주나 | 오늘 쓰는 방법 |
|---|---|---|
| **긍정리뷰 ○○%** | 전체 리뷰 중 긍정 리뷰 비율 | KNU 감성사전 점수 |
| **요약 문장** | 리뷰 전체를 몇 문장으로 정리 | 계산 결과로 문장 채우기 + 대표 리뷰 |
| **항목별 막대** (향, 사용감, 가격, 배송) | 항목마다 좋아요 / 보통이에요 / 아쉬워요 비율 | 항목 단어 **주변**의 감성 점수 |

> ⚠️ 쇼핑몰의 요약 문장은 보통 **생성형 AI(LLM)** 가 직접 글을 씁니다. 오늘은 LLM 없이 **형태소 분석과 감성사전만으로** 만들기 때문에, 요약 문장은 "정해진 틀에 숫자와 단어를 채우는 방식"으로 만듭니다.

## 실습 진행 과정
| 단계 | 할 일 | 확인할 결과 |
|---|---|---|
| 0. 준비 | 설치, 엑셀 읽기, 감성사전 내려받기 | 리뷰 수, 사전 크기 |
| 1. 단어 나누기 | Kiwi로 리뷰를 단어 목록으로 | 단어 목록 |
| 2. 감성사전 다듬기 | ✏️ 상품평에 맞지 않는 단어 고치기 | 고친 사전 |
| 3. 전체 긍정 비율 | 리뷰마다 점수 → 긍정 % | **긍정리뷰 ○○%** |
| 4. 항목별 감성 | ✏️ 항목 단어 정하기 → 항목마다 비율 | 항목별 표 |
| 5. 요약 문장 | 장점·단점 단어와 대표 리뷰 찾기 | 요약 문장 |
| 6. 화면 그리기 | HTML로 요약 화면 만들기 | **완성 화면** |
| 7. 정리 | 결과 직접 확인, 한계, 연습 문제 | — |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.

---
## 0. 준비


In [1]:
# 한국어 형태소 분석기와 엑셀 읽기 도구 설치 (처음 한 번만)
!pip install -q kiwipiepy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 59.0 MB/s eta 0:00:00


Colab 왼쪽 **📁 파일 창**에 `TS_2-4.남성화장품.xlsx`를 끌어다 놓으세요.

In [3]:
# 표 다루기 도구
import pandas as pd

# ===== ✏️ [입력] 엑셀 파일 이름 =====
excel_path = "TS_2-4.남성화장품.xlsx"
# ===================================

# 엑셀에서 상품평 열만 읽고, 빈 리뷰는 빼기
df = pd.read_excel(excel_path, usecols=["상품평"]).dropna()
df["상품평"] = df["상품평"].astype(str)
print("리뷰 수:", len(df))

리뷰 수: 3001


### KNU 한국어 감성사전 내려받기
군산대학교에서 만든 공개 감성사전입니다. 한 줄에 `단어<탭>점수`가 있고, 점수는 **−2(매우 부정) ~ +2(매우 긍정)** 입니다.

>

In [9]:
# 파일 확인 도구와 표 파일 읽기 옵션
import os
import csv

# 올린 파일이 있으면 그 파일, 없으면 GitHub 주소에서 읽기
if os.path.exists("SentiWord_Dict.txt"):
    source = "SentiWord_Dict.txt"
else:
    source = "https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/SentiWord_Dict.txt"


lex = pd.read_csv(source, sep="\t", header=None, names=["단어", "점수"], quoting=csv.QUOTE_NONE).dropna()
# sep="\t"                 : 칸 구분 기호를 쉼표가 아니라 탭 문자로 지정
# header=None              : 파일 첫 줄을 열 이름이 아니라 데이터로 읽음
# names=["단어", "점수"]    : 두 열의 이름을 직접 붙임
# quoting=csv.QUOTE_NONE   : 따옴표(")를 특수 기호가 아닌 일반 글자로 취급
# .dropna()                : 값이 하나라도 비어 있는(NaN) 행을 제거

knu = dict(zip(lex["단어"].str.strip(), lex["점수"].astype(int)))
# lex["단어"].str.strip()  : 각 단어의 앞뒤 공백 문자를 제거
# lex["점수"].astype(int)  : 점수를 실수(float)에서 정수(int)로 변환
# zip(A, B)                : A와 B의 같은 위치 값을 (단어, 점수) 쌍으로 묶음
# dict(...)                : 쌍들을 {단어: 점수} 딕셔너리로 만듦


print("감성사전 크기:", len(knu))   # 딕셔너리에 들어 있는 단어 수 출력



감성사전 크기: 14851


In [47]:


# pd.Series(딕셔너리): 키 → 인덱스(행 이름), 값 → 데이터인 Series를 만듦
# .tail(10)          : 마지막 10행만 표시
print(pd.Series(knu, name="점수").head(10))
print(pd.Series(knu, name="점수").tail(10))

(-;      1
(;_;)   -1
(^^)     1
(^-^)    1
(^^*     1
(^_^)    1
(^_^;   -1
(^o^)    1
(-_-)   -1
(T_T)   -1
Name: 점수, dtype: int64
촉촉하다    1
은은하다    1
가성비     1
끈적이다   -1
트러블    -1
자극     -1
건조하다   -1
당기다    -1
싸다      1
가볍다     1
Name: 점수, dtype: int64


---
## 1. 단어 나누기
Kiwi로 리뷰를 단어 목록으로 바꿉니다.
- 남기는 단어: 명사, 동사, 형용사, 어근(`촉촉`, `은은`)
- **꼭 남기는 말:** `안`, `못`, `별로` (뜻을 뒤집거나 평가하는 말)
- 동사·형용사는 `좋다`처럼 `다`를, 어근은 `촉촉하다`처럼 `하다`를 붙여 **감성사전 모양**에 맞춥니다.

In [48]:
# Kiwi 형태소 분석기
from kiwipiepy import Kiwi
kiwi = Kiwi()

# 잘못 나뉘는 화장품 단어를 한 단어로 등록
for word in ["올인원", "발림성", "흡수력", "유통기한"]:
    kiwi.add_user_word(word, "NNG") #명사

# 형태소 분석 결과를 감성사전 모양의 단어 목록으로 바꾸는 함수
def to_words(tokens):
    words = []
    for t in tokens:
        tag = t.tag                  # 'VA' 동사 같은 태그
        if tag in {"VV", "VA"} or t.form == "않":
            words.append(t.form + "다")            # 동사·형용사·'않'은 '다' 붙이기
        elif tag == "XR":
            words.append(t.form + "하다")          # 어근은 '하다' 붙이기
        elif tag in {"NNG", "NNP"} or t.form in {"안", "못", "별로"}:
            words.append(t.form)                  # 명사와 부정·평가 표현은 그대로
    return words

# 리뷰 전체를 한 번에 분석 (10초 정도 걸림)
df["단어"] = [to_words(tokens) for tokens in kiwi.tokenize(df["상품평"].tolist())]

# 원문과 단어 목록 비교
print(df["상품평"][1])
print(df["단어"][1])

역시 전에 쓰던 게 좋은 것 같아요.
향도 독하고 피부에 자극이 강해서 별로예요.
['전', '쓰다', '좋다', '같다', '향', '독하다', '피부', '자극', '강하다', '별로']


---
## 2. 감성사전 다듬기
KNU 감성사전은 **일상적인 감정 표현** 중심이라 상품평과 맞지 않는 단어가 있습니다.

| 문제 | 예 | 해결 |
|---|---|---|
| 뜻이 여러 개인 단어 | `바르다`(+2): 사전은 "올바르다", 리뷰는 "로션을 바르다" | 0점으로 (무시) |
| 감정과 관계없는 단어 | `받다`, `쓰다`, `많다` | 0점으로 |
| 사전에 없는 단어 | `별로`, `촉촉하다`, `끈적이다` | 직접 추가 |

**✏️ [입력]** 결과를 보고 단어를 더하거나 빼 보세요.

In [49]:
# ===== ✏️ [입력] 감성사전 고치기 =====
# 0점으로 바꿀 단어 목록(리스트)
ignore_words = ["바르다", "크다", "선", "이상", "받다", "많다", "가능", "높다", "걸리다", "쓰다", "들다"]

# 추가하거나 점수를 바꿀 단어: {단어: 점수} 딕셔너리 (점수 범위 −2 ~ +2)
add_words = {"별로": -2, "아쉽다": -1, "불만": -2, "촉촉하다": 1, "은은하다": 1, "가성비": 1, "순하다": 1,
             "끈적이다": -1, "트러블": -1, "자극": -1, "건조하다": -1, "당기다": -1, "싸다": 1, "가볍다": 1}
# =====================================

for w in ignore_words:          # 0점으로 바꿀 단어를 하나씩 꺼냄
    knu[w] = 0                  # 키가 있으면 값을 0으로 바꾸고, 없으면 값 0인 새 키를 추가
knu.update(add_words)           # add_words의 키-값을 knu에 덮어씀 (있으면 값 변경, 없으면 추가)
print("고친 감성사전 크기:", len(knu))   # 고친 뒤 사전의 항목 수 출력

고친 감성사전 크기: 14867


---
## 3. 전체 긍정 비율

### 💡 점수 매기는 규칙
1. 단어마다 감성사전 점수를 더합니다. (사전에 없으면 0점)
2. 바로 앞에 `안`·`못`, 바로 뒤에 `않다`·`없다`가 있으면 **부호를 뒤집습니다.** (`안 좋다` → −2, `불편 없다` → +2)
3. 합계가 **0보다 크면 긍정**, 0이면 중립, 0보다 작으면 부정입니다.

In [38]:
# ──────────────────────────────────────────────
# 부호를 뒤집는 말(부정어) 정의
# ──────────────────────────────────────────────

# 단어 "앞"에 와서 뒤 단어의 감성 점수 부호를 뒤집는 부정어 집합
# 예: "안 좋다" → "좋다"의 점수 부호를 반대로 바꿈
FLIP_BEFORE = {"안", "못"}

# 단어 "뒤"에 와서 앞 단어의 감성 점수 부호를 뒤집는 부정어 집합
# 예: "좋지 않다" → "좋지"의 점수 부호를 반대로 바꿈
FLIP_AFTER = {"않다", "없다"}


# ──────────────────────────────────────────────
# 단어 하나의 감성 점수를 계산하는 함수
# ──────────────────────────────────────────────
# 매개변수
#   words   : 리뷰 하나를 단어 단위로 나눈 리스트 (예: ["음식", "안", "맛있다"])
#   j       : 점수를 계산할 단어의 위치(인덱스)
#   lexicon : 단어 → 감성 점수를 담은 딕셔너리 (감성 사전)
# 반환값
#   j번째 단어의 감성 점수 (정수 또는 실수)
def word_score(words, j, lexicon):
    # 점수를 계산할 j번째 단어를 꺼내 w에 저장
    w = words[j]

    # 현재 단어가 부정어(앞 부정어 또는 뒤 부정어) 자체인지 확인
    if w in FLIP_BEFORE or w in FLIP_AFTER:
        # 부정어 자체는 감성 점수를 갖지 않으므로 0점을 반환하고 함수 종료
        return 0

    # 감성 사전에서 현재 단어의 점수를 조회
    # 사전에 없는 단어이면 기본값 0을 사용
    value = lexicon.get(w, 0)

    # 바로 앞 단어를 가져옴
    # j가 0이면(첫 번째 단어이면) 앞 단어가 없으므로 빈 문자열 ""을 사용
    before = words[j - 1] if j > 0 else ""

    # 바로 뒤 단어를 가져옴
    # j+1이 리스트 길이 이상이면(마지막 단어이면) 뒤 단어가 없으므로 빈 문자열 ""을 사용
    after = words[j + 1] if j + 1 < len(words) else ""

    # 앞 단어가 앞 부정어("안", "못")이거나
    # 뒤 단어가 뒤 부정어("않다", "없다")이면
    if before in FLIP_BEFORE or after in FLIP_AFTER:
        # 감성 점수의 부호를 반대로 바꿈 (양수 → 음수, 음수 → 양수)
        value = -value

    # 최종 계산된 단어 점수를 반환
    return value


# ──────────────────────────────────────────────
# 리뷰별 전체 감성 점수 계산
# ──────────────────────────────────────────────
# df["단어"] 열의 각 행(w)은 리뷰 하나의 단어 리스트
# 각 리뷰에 대해:
#   1) range(len(w))로 0부터 마지막 위치까지 인덱스 j를 차례로 생성
#   2) 각 위치 j마다 word_score(w, j, knu)로 단어 점수를 계산 (knu는 감성 사전)
#   3) sum()으로 해당 리뷰의 모든 단어 점수를 합산
# 이렇게 구한 리뷰별 합계 점수 리스트를 df의 새 열 "점수"에 저장
df["점수"] = [sum(word_score(w, j, knu) for j in range(len(w))) for w in df["단어"]]


# ──────────────────────────────────────────────
# 긍정 리뷰 비율(%) 계산 및 출력
# ──────────────────────────────────────────────
# df["점수"] > 0 : 점수가 0보다 큰 리뷰는 True, 아니면 False인 불리언 Series 생성
# .mean()        : True=1, False=0으로 계산한 평균 → 긍정 리뷰의 비율(0~1)
# * 100          : 비율을 퍼센트(%) 단위로 변환
# round()        : 소수점 이하를 반올림하여 정수로 만듦
positive_rate = round((df["점수"] > 0).mean() * 100)

# f-문자열로 긍정 리뷰 비율을 출력 (예: "긍정리뷰 63%")
print(f"긍정리뷰 {positive_rate}%")

긍정리뷰 75%


---
## 4. 항목별 감성

### 💡 방법: 항목 단어의 "주변"만 보기
"향은 좋은데 **배송이 느려요**" → 리뷰 전체 점수는 0이지만, **향은 좋아요 / 배송은 아쉬워요** 입니다.
그래서 항목 단어(예: `배송`)를 찾고, **그 단어 앞 1개 ~ 뒤 2개 단어**만 보고 점수를 매깁니다.

```
단어 목록:  [향, 좋다, 배송, 느리다, 아쉽다]
"배송" 주변: [좋다, 배송, 느리다, 아쉽다]  →  배송 점수
```

### 💡 항목마다 다른 사전
같은 단어도 항목마다 뜻이 다릅니다.
- `강하다`: 감성사전은 +1이지만, **향이 강하다**는 대개 단점
- `빠르다`: **배송이 빠르다**는 큰 장점 (+2)

그래서 항목마다 **전용 점수**(`lex`)를 덧붙입니다.

**✏️ [입력]** 항목 이름, 찾을 단어(`keywords`), 화면에 보일 이름(`labels`), 항목 전용 점수(`lex`)를 바꿔 보세요.

In [39]:
# ===== ✏️ [입력] 항목 설정 =====
# ASPECTS : 분석할 평가 항목(aspect)들을 담은 딕셔너리
#   키(key)   : 항목 이름 (예: "향", "사용감", "가격", "배송")
#   값(value) : 해당 항목의 설정을 담은 딕셔너리
#       "keywords" : 리뷰에서 이 항목을 언급했다고 판단할 단어들의 집합(set)
#       "labels"   : 결과 0(좋음), 1(보통), 2(아쉬움)에 대응하는 표시용 문구 리스트
#       "lex"      : 이 항목에서만 적용할 단어별 감성 점수 딕셔너리
#                    (기본 감성 사전 knu보다 우선 적용됨)
ASPECTS = {
    # ── "향" 항목 설정 ──
    "향": {
        # 리뷰에 "향", "향기", "냄새" 중 하나가 있으면 향을 언급한 것으로 판단
        "keywords": {"향", "향기", "냄새"},
        # 결과 0 → "좋아요", 1 → "보통이에요", 2 → "아쉬워요"
        "labels": ["좋아요", "보통이에요", "아쉬워요"],
        # 향 항목 전용 점수: "강하다"·"진하다"·"올드"는 -1, "독하다"는 -2, "무난하다"는 +1
        "lex": {"강하다": -1, "진하다": -1, "독하다": -2, "무난하다": 1, "올드": -1},
    },
    # ── "사용감" 항목 설정 ──
    "사용감": {
        # 보습·흡수·발림·끈적임·산뜻함 관련 단어가 있으면 사용감을 언급한 것으로 판단
        "keywords": {"보습", "촉촉하다", "흡수", "흡수력", "발림", "발림성", "끈적이다", "끈적임", "산뜻하다"},
        # 결과 0 → "만족해요", 1 → "보통이에요", 2 → "아쉬워요"
        "labels": ["만족해요", "보통이에요", "아쉬워요"],
        # 사용감 항목 전용 점수: "무겁다"·"번들거리다"·"뻑뻑하다"는 각각 -1
        "lex": {"무겁다": -1, "번들거리다": -1, "뻑뻑하다": -1},
    },
    # ── "가격" 항목 설정 ──
    "가격": {
        # 가격 관련 단어가 있으면 가격을 언급한 것으로 판단
        "keywords": {"가격", "가성비", "저렴하다", "비싸다", "싸다"},
        # 결과 0 → "저렴해요", 1 → "보통이에요", 2 → "비싸요"
        "labels": ["저렴해요", "보통이에요", "비싸요"],
        # 가격 항목 전용 점수는 없음 → 기본 감성 사전(knu) 점수만 사용
        "lex": {},
    },
    # ── "배송" 항목 설정 ──
    "배송": {
        # 배송·택배·도착·포장 단어가 있으면 배송을 언급한 것으로 판단
        "keywords": {"배송", "택배", "도착", "포장"},
        # 결과 0 → "빨라요", 1 → "보통이에요", 2 → "아쉬워요"
        "labels": ["빨라요", "보통이에요", "아쉬워요"],
        # 배송 항목 전용 점수: "빠르다"·"총알"은 +2, "늦다"·"느리다"·"파손"·"찌그러지다"는 -2, "걸리다"는 -1
        "lex": {"빠르다": 2, "늦다": -2, "느리다": -2, "걸리다": -1, "총알": 2, "파손": -2, "찌그러지다": -2},
    },
}
# ===============================


# ──────────────────────────────────────────────
# 리뷰 하나에 대해 특정 항목의 평가 결과를 계산하는 함수
# ──────────────────────────────────────────────
# 매개변수
#   words   : 리뷰 하나를 단어 단위로 나눈 리스트
#   setting : ASPECTS에 정의된 항목 하나의 설정 딕셔너리
# 반환값
#   0    : 좋음 (주변 단어 점수 합계 > 0)
#   1    : 보통 (주변 단어 점수 합계 = 0)
#   2    : 아쉬움 (주변 단어 점수 합계 < 0)
#   None : 이 리뷰에서 해당 항목 키워드가 한 번도 나오지 않음
def aspect_result(words, setting):
    # 기본 감성 사전 knu를 펼친 뒤 항목 전용 사전 setting["lex"]를 펼쳐 하나의 딕셔너리로 합침
    # 같은 단어가 양쪽에 있으면 뒤에 펼친 setting["lex"]의 점수로 덮어씀
    lexicon = {**knu, **setting["lex"]}

    # enumerate로 (위치 i, 단어 w)를 차례로 꺼내
    # 단어가 이 항목의 키워드 집합에 들어 있는 위치 i만 모아 리스트로 만듦
    places = [i for i, w in enumerate(words) if w in setting["keywords"]]

    # 키워드 위치 리스트가 비어 있으면(= 이 항목을 언급하지 않은 리뷰이면)
    if not places:
        # None을 반환하고 함수 종료
        return None

    # 키워드 주변 단어들의 감성 점수를 누적할 변수를 0으로 초기화
    total = 0

    # 키워드가 등장한 각 위치 i에 대해 반복
    for i in places:
        # 키워드 앞 1개 단어부터 키워드 뒤 2개 단어까지의 위치 j를 차례로 생성
        #   시작: max(0, i - 1)          → 리스트 앞쪽 범위를 벗어나지 않도록 0 미만이면 0으로 맞춤
        #   끝  : min(len(words), i + 3) → 리스트 뒤쪽 범위를 벗어나지 않도록 길이를 넘으면 길이로 맞춤
        #   (range의 끝 값은 포함되지 않으므로 실제 범위는 i-1, i, i+1, i+2)
        for j in range(max(0, i - 1), min(len(words), i + 3)):
            # j번째 단어의 점수를 합쳐진 사전(lexicon)으로 계산해 total에 더함
            # (word_score는 앞뒤 부정어가 있으면 점수 부호를 뒤집음)
            total += word_score(words, j, lexicon)

    # total이 양수이면 0(좋음)
    # total이 음수이면 2(아쉬움)
    # total이 0이면 1(보통)을 반환
    return 0 if total > 0 else (2 if total < 0 else 1)


# ──────────────────────────────────────────────
# 항목마다 리뷰별 결과 열 만들기
# ──────────────────────────────────────────────
# ASPECTS의 (항목 이름, 설정)을 하나씩 꺼내 반복
for name, setting in ASPECTS.items():
    # df["단어"]의 각 리뷰 단어 리스트 w에 대해 aspect_result를 호출해 결과 리스트를 만들고
    # 항목 이름(예: "향")을 열 이름으로 하여 df에 새 열로 저장
    # (언급이 없는 리뷰는 None이 들어가며, pandas에서는 NaN으로 표시됨)
    df[name] = [aspect_result(w, setting) for w in df["단어"]]


# ──────────────────────────────────────────────
# 항목별 리뷰 수와 비율(%) 요약표 만들기
# ──────────────────────────────────────────────
# 요약표의 각 행(딕셔너리)을 모아 둘 빈 리스트
rows = []

# ASPECTS의 (항목 이름, 설정)을 하나씩 꺼내 반복
for name, setting in ASPECTS.items():
    # df[name].value_counts() : 해당 항목 열에서 값(0, 1, 2)별 개수를 셈 (NaN은 제외)
    # .reindex([0, 1, 2], fill_value=0) : 순서를 0, 1, 2로 고정하고, 없는 값은 개수 0으로 채움
    counts = df[name].value_counts().reindex([0, 1, 2], fill_value=0)

    # 이 항목을 언급한 전체 리뷰 수 (좋음 + 보통 + 아쉬움)
    total = counts.sum()

    # k = 0(좋음), 1(보통), 2(아쉬움) 순서로 반복
    for k in range(3):
        # 요약표의 한 행을 딕셔너리로 만들어 rows에 추가
        rows.append({
            "항목": name,                                    # 항목 이름 (예: "향")
            "평가": setting["labels"][k],                    # k에 해당하는 표시 문구 (예: "좋아요")
            "리뷰 수": counts[k],                            # k 결과에 해당하는 리뷰 개수
            "비율(%)": round(counts[k] / total * 100),       # 언급 리뷰 중 해당 결과의 비율을 % 정수로 반올림
        })

# rows 리스트(딕셔너리들의 리스트)를 DataFrame으로 변환해 요약표 생성
summary = pd.DataFrame(rows)

# 노트북 셀의 마지막 줄에 변수를 두어 요약표를 화면에 표시
summary

,항목,평가,리뷰 수,비율(%)
0,향,좋아요,812,64
1,향,보통이에요,153,12
2,향,아쉬워요,307,24
3,사용감,만족해요,716,65
4,사용감,보통이에요,101,9
5,사용감,아쉬워요,289,26
6,가격,저렴해요,872,73
7,가격,보통이에요,81,7
8,가격,비싸요,249,21
9,배송,빨라요,354,69


---
## 5. 요약 문장 만들기
LLM 없이 요약 문장을 만들기 위해 **계산 결과를 정해진 틀에 채웁니다.**
1. 긍정 리뷰에만 많이 나온 **명사**(장점) 3개, 부정 리뷰에만 많이 나온 명사(단점) 3개 (`피부`처럼 양쪽에 다 많은 단어는 뺍니다)
2. 항목별로 가장 높은 평가
3. 짧고 점수가 높은 **대표 리뷰** 1개

In [40]:
# 단어 세기 도구
# collections 모듈의 Counter 클래스를 가져옴
# Counter는 항목별 등장 횟수를 딕셔너리 형태로 세어 주는 자료형
from collections import Counter


# ──────────────────────────────────────────────
# 요약에서 제외할 흔한 단어 집합
# ──────────────────────────────────────────────
# 리뷰에 자주 나오지만 장점·단점을 구분하는 데 도움이 되지 않는 단어들을 모아 둔 집합(set)
# top_nouns 함수에서 이 집합에 들어 있는 단어는 개수를 세지 않음
skip = {"제품", "사용", "구매", "구입", "생각", "남편", "신랑", "화장품", "올인원", "로션", "스킨", "때", "것", "만족", "좋다", "별로",
        "마음", "처음", "느낌", "정도", "이번", "기존", "선물", "아빠", "아버지"}


# ──────────────────────────────────────────────
# 리뷰 묶음에서 가장 많이 나온 명사 n개를 반환하는 함수
# ──────────────────────────────────────────────
# 매개변수
#   reviews : 리뷰 문장들이 들어 있는 반복 가능한 객체 (예: pandas Series)
#   n       : 반환할 명사 개수 (기본값 3)
# 반환값
#   등장 횟수가 많은 순서대로 정렬된 명사 문자열 리스트 (최대 n개)
def top_nouns(reviews, n=3):
    # 명사별 등장 횟수를 셀 빈 Counter 객체 생성
    counts = Counter()

    # 리뷰 문장을 하나씩 꺼내 반복
    for text in reviews:
        # kiwi 형태소 분석기로 문장을 형태소(토큰) 단위로 분석하고, 토큰을 하나씩 꺼내 반복
        # 각 토큰 t는 t.form(형태소 문자열), t.tag(품사 태그) 등의 속성을 가짐
        for t in kiwi.tokenize(text):
            # 아래 세 조건을 모두 만족하는 토큰만 개수를 셈
            #   t.tag == "NNG"         : 품사가 일반명사(NNG)인 경우
            #   len(t.form) > 1        : 글자 수가 2글자 이상인 경우 (한 글자 명사 제외)
            #   t.form not in skip     : 제외 단어 집합(skip)에 들어 있지 않은 경우
            if t.tag == "NNG" and len(t.form) > 1 and t.form not in skip:
                # 해당 명사의 등장 횟수를 1 증가
                counts[t.form] += 1

    # counts.most_common(n) : 등장 횟수가 많은 순으로 (단어, 횟수) 튜플 n개를 반환
    # 리스트 컴프리헨션으로 튜플에서 단어(w)만 꺼내 리스트로 만들어 반환 (횟수는 _로 버림)
    return [w for w, _ in counts.most_common(n)]


# ──────────────────────────────────────────────
# 긍정 리뷰·부정 리뷰에서 많이 나온 명사 10개씩 추출
# ──────────────────────────────────────────────
# df["점수"] > 0 인 행(긍정 리뷰)만 골라 "상품평" 열을 꺼낸 뒤
# top_nouns로 가장 많이 나온 명사 10개를 구해 pos_top에 저장
pos_top = top_nouns(df[df["점수"] > 0]["상품평"], 10)

# df["점수"] < 0 인 행(부정 리뷰)만 골라 "상품평" 열을 꺼낸 뒤
# top_nouns로 가장 많이 나온 명사 10개를 구해 neg_top에 저장
# (점수가 정확히 0인 리뷰는 긍정·부정 어디에도 포함되지 않음)
neg_top = top_nouns(df[df["점수"] < 0]["상품평"], 10)


# ──────────────────────────────────────────────
# 한쪽에만 많이 나온 명사를 장점·단점 단어로 선정
# ──────────────────────────────────────────────
# pos_top의 단어 중 neg_top에 없는 단어만 순서대로 남기고, 그중 앞에서부터 3개를 장점 단어로 저장
good_words = [w for w in pos_top if w not in neg_top][:3]

# neg_top의 단어 중 pos_top에 없는 단어만 순서대로 남기고, 그중 앞에서부터 3개를 단점 단어로 저장
bad_words = [w for w in neg_top if w not in pos_top][:3]

# 장점 단어 리스트를 출력
print("장점 단어:", good_words)

# 단점 단어 리스트를 출력
print("단점 단어:", bad_words)


# ──────────────────────────────────────────────
# 대표 리뷰 선정: 30~80자 리뷰 중 점수가 가장 높은 리뷰
# ──────────────────────────────────────────────
# df["상품평"].str.len()        : 각 리뷰의 글자 수를 계산
# .between(30, 80)              : 글자 수가 30 이상 80 이하이면 True (양 끝 값 포함)
# df[...]                       : True인 행만 골라 short에 저장
short = df[df["상품평"].str.len().between(30, 80)]

# short.sort_values("점수", ascending=False) : "점수" 열 기준 내림차순 정렬 (높은 점수가 맨 위)
# ["상품평"]                                  : 정렬된 결과에서 "상품평" 열만 선택
# .iloc[0]                                    : 첫 번째 행(점수가 가장 높은 리뷰)의 문장을 꺼냄
# .replace("\n", " ")                         : 리뷰 안의 줄바꿈 문자를 공백으로 바꿔 한 줄로 만듦
best_review = short.sort_values("점수", ascending=False)["상품평"].iloc[0].replace("\n", " ")

# 대표 리뷰를 출력
print("대표 리뷰:", best_review)

장점 단어: ['면도', '가성비', '용기']
단점 단어: ['품질', '건조', '불편']
대표 리뷰: 가성비 좋습니다. 용량도 크고 거품도 많이 나서 좋습니다. 품질도 만족스러워요. 사용하기도 편합니다. 만족스럽습니다


In [42]:
# ──────────────────────────────────────────────
# 항목마다 비율이 가장 높은 평가를 문장 조각으로 만들기
# ──────────────────────────────────────────────
# 항목별 문장 조각을 모아 둘 빈 리스트
parts = []

# ASPECTS 딕셔너리의 키(항목 이름)를 하나씩 꺼내 반복
for name in ASPECTS:
    # summary["항목"] == name                  : 요약표에서 현재 항목에 해당하는 행만 True
    # summary[...]                             : 해당 항목의 행 3개(좋음·보통·아쉬움)만 선택
    # .sort_values("비율(%)", ascending=False) : 비율 기준 내림차순 정렬 (높은 비율이 맨 위)
    # .iloc[0]                                 : 첫 번째 행(비율이 가장 높은 평가)을 Series로 꺼냄
    top = summary[summary["항목"] == name].sort_values("비율(%)", ascending=False).iloc[0]

    # f-문자열로 문장 조각을 만들어 parts에 추가
    #   {name}             : 항목 이름을 넣음
    #   {(name)}           : 괄호로 감싼 name을 한 번 더 넣음 (괄호는 값에 영향을 주지 않음)
    #   '{top['평가']}'    : 가장 비율이 높은 평가 문구를 작은따옴표로 감싸 넣음
    #   {top['비율(%)']}%  : 해당 평가의 비율 숫자 뒤에 % 기호를 붙여 넣음
    parts.append(f"{name}{(name)} '{top['평가']}' {top['비율(%)']}%")


# ──────────────────────────────────────────────
# 정해진 틀에 값을 채워 요약 문장 완성
# ──────────────────────────────────────────────
# 괄호 안에 여러 f-문자열을 나란히 쓰면 파이썬이 하나의 문자열로 이어 붙임
summary_text = (
    # len(df):,     : 전체 리뷰 수를 천 단위 쉼표 형식으로 표시 (예: 1,234)
    # positive_rate : 앞 셀에서 계산한 긍정 리뷰 비율(%)
    f"리뷰 {len(df):,}건 중 {positive_rate}%가 긍정적이에요. "

    # ', '.join(parts) : 항목별 문장 조각들을 ", "로 연결해 하나의 문자열로 만듦
    f"{', '.join(parts)}로 나타났어요. "

    # ', '.join(good_words) : 장점 단어 리스트를 ", "로 연결
    f"좋은 점으로는 {', '.join(good_words)}에 대한 이야기가 많고, "

    # ', '.join(bad_words) : 단점 단어 리스트를 ", "로 연결
    f"아쉬운 점으로는 {', '.join(bad_words)}에 대한 이야기가 많아요. "

    # \" : 문자열 안에 큰따옴표를 넣기 위한 이스케이프 문자
    # best_review : 앞 셀에서 선정한 대표 리뷰 문장
    f"대표 리뷰: \"{best_review}\""
)

# 완성된 요약 문장을 출력
print(summary_text)

리뷰 3,001건 중 75%가 긍정적이에요. 향향 '좋아요' 64%, 사용감사용감 '만족해요' 65%, 가격가격 '저렴해요' 73%, 배송배송 '빨라요' 69%로 나타났어요. 좋은 점으로는 면도, 가성비, 용기에 대한 이야기가 많고, 아쉬운 점으로는 품질, 건조, 불편에 대한 이야기가 많아요. 대표 리뷰: "가성비 좋습니다. 용량도 크고 거품도 많이 나서 좋습니다. 품질도 만족스러워요. 사용하기도 편합니다. 만족스럽습니다"


---
## 6. 요약 화면 그리기
HTML로 쇼핑몰과 비슷한 화면을 만듭니다. HTML은 웹페이지를 만드는 언어로, 노트북 안에서 `display(HTML(...))`로 바로 보여 줄 수 있습니다.
첫 번째 평가 막대는 **빨간색**, 나머지는 **회색**입니다.

In [43]:
# 노트북에 HTML을 보여 주는 도구
from IPython.display import HTML, display

# 막대 한 줄(평가 이름, 막대, 리뷰 수, 비율)의 HTML을 만드는 함수
def bar_html(label, count, percent, color):
    return f"""
    <div style="margin:10px 0">
      <div style="font-size:14px">{label}</div>
      <div style="display:flex;align-items:center;gap:8px">
        <div style="flex:1;background:#eee;height:6px;border-radius:3px">
          <div style="width:{percent}%;background:{color};height:6px;border-radius:3px"></div>
        </div>
        <span style="color:#999;font-size:12px">({count:,})</span>
        <b style="width:40px;text-align:right">{percent}%</b>
      </div>
    </div>"""

# 항목 하나(제목 + 막대 3줄)의 HTML을 만드는 함수
def aspect_html(name):
    rows = summary[summary["항목"] == name].reset_index(drop=True)
    bars = "".join(bar_html(r["평가"], r["리뷰 수"], r["비율(%)"], "#f05050" if i == 0 else "#bbb") for i, r in rows.iterrows())
    return f'<div style="flex:1;min-width:180px"><b>{name}</b>{bars}</div>'

# 전체 화면 HTML 조립
page = f"""
<div style="font-family:sans-serif;max-width:900px;padding:16px;background:#fff;color:#222">
  <div style="text-align:center;font-size:18px;font-weight:bold">리뷰 분석 요약</div>
  <div style="text-align:center;color:#f05050;margin:8px 0">긍정리뷰 <span style="font-size:28px;font-weight:bold">{positive_rate}%</span></div>
  <div style="background:#f5f5f5;padding:14px;border-radius:6px;line-height:1.6;font-size:14px">{summary_text}</div>
  <div style="display:flex;gap:24px;flex-wrap:wrap;margin-top:16px">{"".join(aspect_html(n) for n in ASPECTS)}</div>
  <div style="color:#999;font-size:11px;margin-top:8px">* Kiwi 형태소 분석 + KNU 한국어 감성사전으로 계산한 결과입니다.</div>
</div>"""

# 화면에 보여 주기
display(HTML(page))

In [44]:
# 만든 화면을 HTML 파일로 저장 (내려받아 웹 브라우저로 열 수 있음)
with open("review_summary.html", "w", encoding="utf-8") as f:
    f.write(page)
print("저장 완료: review_summary.html")

저장 완료: review_summary.html


---
## 7. 정리

### ✍️ 결과 직접 확인하기
계산 결과가 맞는지 **원래 리뷰를 읽어서** 확인해야 합니다.
**✏️ [입력]** 항목 이름과 평가 번호(0: 좋음, 1: 보통, 2: 아쉬움)를 바꿔 보세요.

In [46]:
# ===== ✏️ [입력] 확인할 항목과 평가 =====
# 확인할 항목 이름 (ASPECTS에 정의된 키 중 하나: "향", "사용감", "가격", "배송")
check_aspect = "향"

# 확인할 평가 결과 번호
#   0 : 좋음 (예: 향 → "좋아요")
#   1 : 보통 (예: 향 → "보통이에요")
#   2 : 아쉬움 (예: 향 → "아쉬워요")
check_result = 2
# =======================================


# ──────────────────────────────────────────────
# 지정한 항목·평가로 분류된 리뷰 중 5개를 무작위로 뽑기
# ──────────────────────────────────────────────
# df[check_aspect]                 : 확인할 항목의 결과 열 (예: df["향"])
# == check_result                  : 결과 값이 지정한 평가 번호와 같으면 True
# df[...]                          : True인 행(해당 평가로 분류된 리뷰)만 선택
# .sample(5, random_state=1)       : 선택된 행 중 5개를 무작위로 추출
#                                    random_state=1 은 난수 시드를 고정해 실행할 때마다 같은 5개가 뽑히게 함
sample = df[df[check_aspect] == check_result].sample(5, random_state=1)


# ──────────────────────────────────────────────
# 뽑은 리뷰를 한 줄씩 출력
# ──────────────────────────────────────────────
# sample의 "상품평" 열에서 리뷰 문장을 하나씩 꺼내 반복
for text in sample["상품평"]:
    # text[:100]          : 리뷰 문장의 앞 100글자만 잘라냄
    # .replace("\n", " ") : 잘라낸 문장 안의 줄바꿈 문자를 공백으로 바꿔 한 줄로 만듦
    # "-" 기호를 앞에 붙여 목록 형태로 출력
    print("-", text[:100].replace("\n", " "))

- 펌프 방식이라서 구매했는데 용기에서 내용물이 흘러나와서 불편해요. 끈적임도 있고 산뜻하지도 않아요. 남성 스킨 향이 나서 별로예요.
- 남편이 여름에 쓰는 화장품중에 젤루 별로라고 하네요. 끈적이고 향도 별로라고 해서 재구매 의사 없습니다.
- 처음 사보는 브랜드 제품입니다. 향이 생각보다 좋지 않네요. 유분을 잘 잡아주지 못해서 피부가 번들거립니다.
- 가성비 나쁘지 않지만 멘톨 성분이 강하진 않아서 별로네요. 향이 좀 더 화한 느낌이었으면 더 좋을 것 같습니다. 그리고 샘플이 없어서 아쉬워요.
- 여러 제품 사용해 봤지만 이 제품도 끈적임이 있고 향이 약해서 아쉬워요. 양이 넉넉해서 오래 사용할 수는 있겠어요. 그런데 용기에서 내용물이 잘 나오지는 않네요. 이 제품도 만족스


### ✅ 한 줄 정리
**리뷰 전체 점수로 긍정 비율을 구하고, 항목 단어 주변의 점수로 항목별 평가를 구해 요약 화면을 만들었다.**

### 🤔 생각해 볼 점 (한계)
| 한계 | 예 | 더 나은 방법 |
|---|---|---|
| **앞 1개~뒤 2개만 봄** | "향은 좋은데 끈적임이 있어 아쉬워요"에서 다른 항목의 말이 섞일 수 있음 | 문장 구조(주어-서술어)를 분석하는 방법 |
| **사전에 없는 표현** | "향이 오래가요", "총알배송" | 사전에 단어 추가, 항목 전용 점수 |
| **정답과 비교하지 않음** | 긍정 %가 정확한지 모름 | 사람이 채점한 데이터와 비교 |





**참고:** [KNU 한국어 감성사전 (GitHub)](https://github.com/park1200656/KnuSentiLex) · [Kiwi (kiwipiepy)](https://github.com/bab2min/kiwipiepy)